In [ ]:
import os
import random

def load_dialect(dialect_path):
    samples = []

    # read ALL transcript files
    transcript = {}

    for f in os.listdir(dialect_path):
        if f.lower().endswith("_text.txt"):
            with open(os.path.join(dialect_path, f), encoding="utf-8") as t:
                for line in t:
                    line = line.strip()
                    if not line:
                        continue

                    parts = line.split(maxsplit=1)
                    if len(parts) != 2:
                        continue

                    audio_id, text = parts
                    wav_name = audio_id + ".wav"
                    transcript[wav_name] = text.strip()

    # collect all wav files
    all_wavs = []
    for root, _, files in os.walk(dialect_path):
        for f in files:
            if f.lower().endswith(".wav"):
                all_wavs.append(os.path.join(root, f))

    random.shuffle(all_wavs)

    for wav in all_wavs:
        fname = os.path.basename(wav)
        if fname in transcript:
            samples.append({
                "audio": wav,
                "text": transcript[fname]
            })
        if len(samples) >= max_files:
            break

    return samples


In [ ]:
all_samples = []

root = "Dataset"
for dialect in [
    "Central_Dialect",
    "Northern_Dialect",
    "Southern_Dialect",
    "Western_Dialect"
]:
    s = load_dialect(os.path.join(root, dialect), max_files=100)
    print(dialect, "→", len(s))
    all_samples.extend(s)



In [3]:
print(all_samples[0])


{'audio': 'Dataset/Central_Dialect/SP19_THA_audio/SP19_THA_F_49.wav', 'text': 'போயி நீங்க பாட்டுக்கு தூங்க போயிட்டிய அதுவ அநியாயம் பண்ணி வச்சுருக்கு அடுப்படில'}


In [4]:
from datasets import Dataset, Audio

ds = Dataset.from_list(all_samples)
ds = ds.cast_column("audio", Audio(sampling_rate=16000))


/home/pranesh/Desktop/Dialect /venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

model_name = "openai/whisper-tiny"

processor = WhisperProcessor.from_pretrained(
    model_name,
    language="ta",
    task="transcribe"
)

model = WhisperForConditionalGeneration.from_pretrained(model_name)

# IMPORTANT for fine-tuning
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []


Loading weights: 100%|██████████| 167/167 [00:00<00:00, 210.82it/s, Materializing param=model.encoder.layers.3.self_attn_layer_norm.weight]  


In [6]:
def prepare(batch):
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(
        audio["array"],
        sampling_rate=16000
    ).input_features[0]

    batch["labels"] = processor.tokenizer(
        batch["text"]
    ).input_ids
    return batch

ds = ds.map(prepare, remove_columns=ds.column_names)


Map: 100%|██████████| 400/400 [00:34<00:00, 11.64 examples/s]


In [7]:
import torch

def data_collator(features):
    input_features = torch.tensor([f["input_features"] for f in features])
    labels = torch.nn.utils.rnn.pad_sequence(
        [torch.tensor(f["labels"]) for f in features],
        batch_first=True,
        padding_value=-100
    )
    return {
        "input_features": input_features,
        "labels": labels
    }


In [10]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir="./whisper_tamil_ft",
    per_device_train_batch_size=1,
    learning_rate=1e-5,
    max_steps=200,
    save_strategy="no",   # ⬅️ disables checkpoint saving
    logging_steps=10,
    report_to="none"
)




In [11]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds,
    data_collator=data_collator,
)



In [12]:
from transformers import GenerationConfig

model.generation_config = GenerationConfig.from_model_config(model.config)

# remove generation params from model.config
if hasattr(model.config, "suppress_tokens"):
    del model.config.suppress_tokens


In [13]:
trainer.train()

Step,Training Loss
10,0.671118
20,0.674445
30,0.596522
40,0.603926
50,0.747453
60,0.668312
70,0.682552
80,0.540957
90,0.562208
100,0.593218


TrainOutput(global_step=200, training_loss=0.6263997650146484, metrics={'train_runtime': 439.9869, 'train_samples_per_second': 0.455, 'train_steps_per_second': 0.455, 'total_flos': 4923777024000000.0, 'train_loss': 0.6263997650146484, 'epoch': 0.5})

In [14]:
model.save_pretrained("whisper-tamil-finetuned")
processor.save_pretrained("whisper-tamil-finetuned")


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.65it/s]


['whisper-tamil-finetuned/processor_config.json']

In [15]:
import torchaudio, torch

wav, sr = torchaudio.load(all_samples[0]["audio"])
inputs = processor(wav.squeeze(), sampling_rate=sr, return_tensors="pt")

with torch.no_grad():
    pred = model.generate(inputs.input_features)

print(processor.tokenizer.decode(pred[0], skip_special_tokens=True))


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.


போயி நீங்க பாட்டு


In [16]:
import torch, torchaudio

model.eval()

results = []

for i in range(20):
    sample = all_samples[i]
    wav_path = sample["audio"]

    wav, sr = torchaudio.load(wav_path)

    inputs = processor(
        wav.squeeze(),
        sampling_rate=sr,
        return_tensors="pt"
    )

    with torch.no_grad():
        pred_ids = model.generate(
            inputs.input_features,
            max_new_tokens=128
        )

    text = processor.tokenizer.decode(
        pred_ids[0],
        skip_special_tokens=True
    )

    print(f"\nSample {i+1}")
    print("Audio:", wav_path)
    print("Transcription:", text)

    results.append({
        "audio": wav_path,
        "transcription": text
    })


Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Sample 1
Audio: Dataset/Central_Dialect/SP19_THA_audio/SP19_THA_F_49.wav
Transcription: போயி நீங்க பாட்டுக்கு தூங்க போயிட்டியே அது வாணியா இம்பண்ணி மச்சிருக்கா, அடுப்படி இல்ல.


Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Sample 2
Audio: Dataset/Central_Dialect/SP4_THA_audio/SP4_THA_M_51.wav
Transcription: தீனி கேட்டது அவன் கல்ல அரிசி தப்பாவ போட்டு ஒடச்சது அவன் நான் என் சுத்தம் பண்ணானும்


Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


Sample 18
Audio: Dataset/Central_Dialect/SP5_THA_audio/SP5_THA_M_1.wav
Transcription: எனுன் நைச்சா காலைலையாடு புடிக்குள்ளும் தீங்க.


Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Sample 19
Audio: Dataset/Central_Dialect/SP15_THA_audio/SP15_THA_F_2.wav
Transcription: அய் இந்த போடவமடி பையடுத்துவிட்ட.


Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Sample 20
Audio: Dataset/Central_Dialect/SP6_THA_audio/SP6_THA_M_15.wav
Transcription: கத்துரிக்கா இந்த கத்துரிக்காதான் இருக்கான் நாட்டு கத்துரிக்கா இல்லைங்களா?


In [19]:
import os
import torch
import torchaudio
import pandas as pd

model.eval()

TEST_DIR = "Test_Dialect"
OUTPUT_CSV = "test_dialect_asr.csv"

# 🔹 Load already processed files
if os.path.exists(OUTPUT_CSV):
    df_done = pd.read_csv(OUTPUT_CSV)
    done_files = set(df_done["audio_file"].astype(str).tolist())
    print(f"Resuming… already done: {len(done_files)} files")
else:
    done_files = set()
    pd.DataFrame(columns=["audio_file", "transcription"]).to_csv(
        OUTPUT_CSV, index=False, encoding="utf-8"
    )

sample_no = len(done_files) + 1

for root, _, files in os.walk(TEST_DIR):
    for f in sorted(files):
        if not f.lower().endswith(".wav"):
            continue

        # 🔹 SKIP if already processed
        if f in done_files:
            continue

        wav_path = os.path.join(root, f)

        try:
            wav, sr = torchaudio.load(wav_path)

            # Stereo → mono
            if wav.dim() == 2:
                wav = wav.mean(dim=0)

            # Skip extremely short audio
            if wav.numel() < 400:
                print(f"⚠️ Skipped short audio: {f}")
                continue

            # Resample to 16k
            if sr != 16000:
                wav = torchaudio.functional.resample(wav, sr, 16000)

            inputs = processor(
                wav,
                sampling_rate=16000,
                return_tensors="pt",
                padding=True
            )

            with torch.no_grad():
                pred_ids = model.generate(
                    inputs.input_features,
                    max_new_tokens=128,
                    max_length=None
                )

            text = processor.tokenizer.decode(
                pred_ids[0],
                skip_special_tokens=True
            )

            # 🔹 Print
            print(f"\nSample {sample_no}")
            print("Audio:", f)
            print("Transcription:", text)

            # 🔹 Append immediately
            pd.DataFrame([[f, text]], columns=["audio_file", "transcription"]).to_csv(
                OUTPUT_CSV,
                mode="a",
                header=False,
                index=False,
                encoding="utf-8"
            )

            done_files.add(f)
            sample_no += 1

        except Exception as e:
            print(f"❌ Skipped {f} due to error: {e}")
            continue

print("\n✅ RESUME COMPLETE")
print(f"Final CSV → {OUTPUT_CSV}")
print(f"Total files transcribed: {len(done_files)}")


Resuming… already done: 98 files

Sample 99
Audio: test_0099.wav
Transcription: இந்தைக்கு கால கட்டத்தல போத்தையன் இவிப்போ சமியுப்புத்தல வந்து போட்டும்முன்னு பாத்தைங்கன்னும் விடுவுவேல. அதுல வந்து அரும்மியா சூரி நடிச்சுரு

Sample 100
Audio: test_0100.wav
Transcription: இருந்தாலாம் யிரோட்டல எல்லா தூனி வையலோம் நாமா குசாட்ட காண்டு தெரி மெரி செல்ல தூனி முன்னி எல்லாவாங்குங்க ஏன்ல மேன் யிரோட்ல இந்தரண்டுந்தான் அ

Sample 101
Audio: test_0101.wav
Transcription: எதையனும் திரைவியும்னா, ரோம் கோரும் நல்ல புடிக்கும் என்ன? பாதுகாப்போயிட்டு வருனுங்க. என்ன அம்முள்ளும் பாதுகாப்போதுகாப்போதுக்கும் என்ன? எதுரு

Sample 102
Audio: test_0102.wav
Transcription: அதே மரி பச்சு போயிருங்க பட்ட்ட்கப்பு ரு இதலாராவோ பூடிங்க. ரதே மரி எது ஆசாய்வுத்தல வந்து பாத்துங்க மாட்ட்டன் சிக்கன்னும் மரி அம்போபூடிங்க. �

Sample 103
Audio: test_0103.wav
Transcription: மத்த மாடுத்துல வழாண்ட பாருன் நென்னுட்டு வாதியேன்.

Sample 104
Audio: test_0104.wav
Transcription: மக்கு நாளிக்கேன்னுக்கு வினுவேசுட்டு அக்குதான் பீ.

Sample 105
Audio: tes

In [1]:
import pandas as pd
import re

INPUT_CSV = "test_dialect_asr.csv"        # your current CSV
OUTPUT_TXT = "submission.txt"             # final submission file

df = pd.read_csv(INPUT_CSV)

def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Remove bracketed noise
    text = re.sub(r"\[.*?\]", "", text)

    # Collapse long repeated characters (keep one)
    text = re.sub(r"(.)\1{4,}", r"\1", text)

    # Collapse repeated words
    text = re.sub(r"\b(\w+)( \1\b)+", r"\1", text)

    # Keep Tamil + English + basic punctuation
    text = re.sub(r"[^\w\s\u0B80-\u0BFF.,?']", "", text)

    # Normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

with open(OUTPUT_TXT, "w", encoding="utf-8") as f:
    for _, row in df.iterrows():
        audio_file = row["audio_file"]
        text = clean_text(row["transcription"])

        # remove .wav extension → test_0001
        test_id = audio_file.replace(".wav", "")

        # ALWAYS write the row (even if text is empty)
        f.write(f"{test_id} {text}\n")

print("✅ Submission file created:", OUTPUT_TXT)
print("Total lines written:", len(df))


✅ Submission file created: submission.txt
Total lines written: 579
